In [1]:
import base64
import json
import uuid
import requests
import os
import pandas as pd

In [2]:
def generate_audio(
        uid, appid, access_token, 
        text, filepath, 
        voice_type="BV701_V2_streaming", rate=44100, encoding="wav",
        speed_ratio=1.0, volume_ratio=1.0, pitch_ratio=1.0, 
        language="cn",
        verbose=True
    ):
    cluster = "volcano_tts"

    host = "openspeech.bytedance.com"
    api_url = f"https://{host}/api/v1/tts"

    header = {"Authorization": f"Bearer;{access_token}"}

    request_json = {
        "app": {
            "appid": appid,
            "token": "access_token",
            "cluster": cluster
        },
        "user": {
            "uid": uid
        },
        "audio": {
            "voice_type": voice_type,
            "encoding": encoding,
            "rate": rate,
            "speed_ratio": speed_ratio,
            "volume_ratio": volume_ratio,
            "pitch_ratio": pitch_ratio,
            "language": language
        },
        "request": {
            "reqid": str(uuid.uuid4()),
            "text": text,
            "text_type": "plain",
            "operation": "query",
            "with_frontend": 1,
            "frontend_type": "unitTson"

        }
    }
    try:
        resp = requests.post(api_url, json.dumps(request_json), headers=header)
        if verbose:
            print(f"Resp Body: \n{resp.json()}")
        if "data" in resp.json():
            data = resp.json()["data"]
            file_to_save = open(filepath, "wb")
            file_to_save.write(base64.b64decode(data))
    except Exception as e:
        e.with_traceback()
    return resp.json()

In [ ]:
# Define some configurations
audio_paramters = {
    "uid" : input("Please input your uid: "),
    "appid" : input("Please input your appid: "),
    "access_token" : input("Please input your access_token: "),
    "voice_type" : "BV002_streaming",
    "rate" : 44100,
    "encoding" : "wav",
    "speed_ratio" : 1.0,
    "volume_ratio" : 1.0,
    "pitch_ratio" : 1.0,
    "language" : "cn",
    "verbose" : False
}

save_folder = "./experiment/paradigm/stimulus/aud/volcano_tts/aud_origin"
if not os.path.exists(save_folder):
    os.makedirs(save_folder, exist_ok=True)

qualified_block_indices = [4, 5, 6, 7, 10, 12, 13]

# Current directory
current_directory = os.getcwd()

In [ ]:
# Read the sentences
df = pd.read_csv(os.path.join(os.path.dirname(current_directory), "experiment/materials.tsv"), 
                 encoding="utf-8", sep="\t")

sentences = list()
for index, row in df.iterrows():
    sentences.append(
        "。".join([row["Conj1"], row["Sentence1"], 
                   row["Conj2"], row["Sentence2"]]) + "。"
    )

# Generate the audio files
for i in range(len(qualified_block_indices)):
    for j in range(8):
        text = sentences[qualified_block_indices[i] * 8 + j]
        print(f"Generating: {text}")
        filepath = os.path.join(save_folder, f"topic-{i + 1}_sentence-{j + 1}.wav")
        resp = generate_audio(
            text=text,
            filepath=filepath,
            **audio_paramters
        )
        with open(os.path.join(save_folder, f"topic-{i + 1}_sentence-{j + 1}.json"), "w", encoding="utf-8") as f:
            json.dump(resp, f, ensure_ascii=False)